# Pydantic Example

In [ ]:
# Pydantic Demo

from pydantic import BaseModel, EmailStr, ValidationError, Field
from typing import Optional

class User(BaseModel):
    name: str = Field(min_length=3, default='John Doe')
    email: EmailStr
    age: Optional[int] = Field(None, lt=60, gt=18, description="Age must be between 18 and 60")
    address: Optional[str] = None


user = User(
    name="John",          # too short
    age=25,             # invalid age
    email="john@example.com"  # invalid email
)
print(user.name)

try:
    user = User(
        name="Al",          # too short
        age=-5,             # invalid age
        email="not-an-email"  # invalid email
    )
except ValidationError as e:
    print("Validation Error")

# With Pydantic Structured Output

In [ ]:
# Structured Output Pydantic Demo
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional, Literal

load_dotenv()

model = init_chat_model(model=os.getenv("MODEL_NAME"), model_provider=os.getenv("MODEL_PROVIDER"))

class UserInfo(BaseModel):
    name: str = Field(min_length=3, description="Name of the user. Must be at least 3 characters long.")
    email: EmailStr
    age: Optional[int] = Field(None, lt=60, gt=18, description="Age must be between 18 and 60")
    hobbies: Optional[list[str]] = Field(default_factory=list, description="List of user's hobbies")
    address: Optional[str] = Field(None, description="User's address")
    gender: Literal["male", "female"] = Field(description="Gender of the user. Must be either 'male' or 'female'")

structured_model = model.with_structured_output(UserInfo)
result = structured_model.invoke(
    "Rahul Sharma is a 28 year old male software engineer. His email is rahul.sharma@gmail.com. He enjoys hiking and playing guitar. He lives at 123 Main St, Anytown, USA."
)

print(result)

# With Pydantic JSON Output

In [ ]:
# Structured Output JSON Demo
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

model = init_chat_model(model=os.getenv("MODEL_NAME"), model_provider=os.getenv("MODEL_PROVIDER"))

json_schema = {
  "title": "UserInfo",
  "type": "object",
  "properties": {
    "name": {
      "type": "string",
      "minLength": 3,
      "description": "Full name of the user"
    },
    "email": {
      "type": "string",
      "format": "email",
      "description": "User email address"
    },
    "age": {
      "type": ["integer", "null"],
      "exclusiveMinimum": 18,
      "exclusiveMaximum": 60,
      "description": "Age between 18 and 60"
    },
    "gender": {
      "type": "string",
      "enum": ["male", "female"],
      "description": "Gender of the user"
    },
    "hobbies": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "List of hobbies",
      "default": []
    },
    "address": {
      "type": ["string", "null"],
      "description": "User's address"
    }
  },
  "required": ["name", "email", "gender"]
}

structured_model = model.with_structured_output(json_schema)

prompt = PromptTemplate.from_template(
"""
Extract user information.

Text: {text}
"""
)
prompt = prompt.invoke({ 'text': "Rahul Sharma is a 12 year old male software engineer. His email is rahul.sharma@gmail.com. He enjoys hiking and playing guitar. He lives at 123 Main St, Anytown, USA."})
result = structured_model.invoke(prompt)

print(result)